# Chapter 10: SFT Best Practices — Training the GAS Base Model

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# !pip install numpy matplotlib

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Implement **completion-only masking** to prevent training on prompt tokens
- Understand **sequence packing** to maximize GPU utilization
- Implement the **pass@k unbiased estimator** for code/reasoning evaluation
- Apply pass@k to evaluate the **GAS Advisor** on medical test cases
- Know the key SFT best practices that prevent common training failures

---
## 📖 2. Theory Recap

### Supervised Fine-Tuning (SFT)

SFT trains a model to imitate high-quality demonstrations:

$$\mathcal{L}_{\text{SFT}} = -\sum_{t=1}^{T} \log P_\theta(y_t | x, y_{<t})$$

### Key Best Practices

**1. Completion-Only Masking**  
Only compute loss on the response tokens, not the prompt. This prevents the model from "wasting" gradient updates on tokens it shouldn't be generating.

**2. Sequence Packing**  
Instead of padding short sequences to max length, pack multiple examples into one sequence. This eliminates wasted computation on padding tokens.

**3. pass@k Evaluation**  
For tasks with multiple valid solutions, evaluate by sampling k responses and checking if any is correct. The unbiased estimator avoids overestimating performance.

### Common SFT Failures

| Failure | Cause | Fix |
|---------|-------|-----|
| Repetition | Training on padding tokens | Completion-only masking |
| Forgetting | High learning rate | Cosine schedule, low LR |
| Short responses | Length penalty | Remove length bias |
| GPU waste | Padding | Sequence packing |
| Overfit | Small dataset | Data augmentation, early stopping |

---
## 🔨 3. Build It

### 3.1 Completion-Only Masking

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from typing import List, Tuple

def apply_completion_mask(input_ids: List[int], response_start_idx: int) -> List[int]:
    """
    Apply completion-only masking: set prompt tokens to -100 (ignored in loss).
    
    In PyTorch CrossEntropyLoss, label=-100 is ignored.
    This ensures we only train on the response, not the prompt.
    
    Args:
        input_ids: token IDs for the full sequence [prompt + response]
        response_start_idx: index where the response begins
    
    Returns:
        labels: -100 for prompt tokens, actual token IDs for response tokens
    """
    labels = [-100] * response_start_idx + input_ids[response_start_idx:]
    return labels

# Concrete example with a medical prompt
# Simulated tokenization (in practice, use a real tokenizer)
prompt_tokens = [1, 450, 7027, 338, 29871, 29906, 29947, 29900, 29889, 1724, 1818, 306, 437, 29973, 2]  # 15 tokens
response_tokens = [306, 1818, 1423, 363, 413, 1187, 2873, 29892, 6958, 4902, 29892, 13]  # 12 tokens

full_sequence = prompt_tokens + response_tokens
response_start = len(prompt_tokens)

labels = apply_completion_mask(full_sequence, response_start)

print("Completion-Only Masking Example")
print("=" * 55)
print(f"\nPrompt:   '{' '.join(str(t) for t in prompt_tokens)}'")
print(f"Response: '{' '.join(str(t) for t in response_tokens)}'")
print(f"\nFull sequence length: {len(full_sequence)} tokens")
print(f"Response starts at:   index {response_start}")
print(f"\nLabels:")
print(f"  Prompt tokens (masked):   {labels[:response_start]}")
print(f"  Response tokens (active): {labels[response_start:]}")
print(f"\n✅ {response_start} prompt tokens masked (-100 = ignored in loss)")
print(f"✅ {len(full_sequence) - response_start} response tokens contribute to loss")

# Visualize
fig, ax = plt.subplots(figsize=(14, 2.5))
colors = ['#e74c3c' if l == -100 else '#2ecc71' for l in labels]
for i, (token, color) in enumerate(zip(full_sequence, colors)):
    ax.bar(i, 1, color=color, edgecolor='white', linewidth=0.5)
    ax.text(i, 0.5, str(token)[:4], ha='center', va='center', fontsize=7, color='white', fontweight='bold')

ax.axvline(x=response_start - 0.5, color='black', linewidth=3, linestyle='--')
ax.text(response_start/2, 1.1, 'PROMPT (masked, -100)', ha='center', fontsize=10, color='#e74c3c', fontweight='bold')
ax.text(response_start + (len(full_sequence)-response_start)/2, 1.1, 'RESPONSE (active)', ha='center', fontsize=10, color='#2ecc71', fontweight='bold')
ax.set_xlim(-0.5, len(full_sequence) - 0.5)
ax.set_ylim(0, 1.4)
ax.set_xlabel('Token Position')
ax.set_title('Completion-Only Masking: Only Response Tokens Contribute to Loss', fontsize=12, fontweight='bold')
ax.set_yticks([])
plt.tight_layout()
plt.savefig('completion_masking.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.2 Sequence Packing

In [ ]:
def pack_sequences(
    sequences: List[List[int]],
    max_length: int,
    pad_token_id: int = 0,
    eos_token_id: int = 2
) -> Tuple[List[List[int]], List[List[int]]]:
    """
    Pack multiple short sequences into fixed-length bins.
    Eliminates padding waste.
    
    Returns:
        packed_sequences: list of packed sequences (each length = max_length)
        attention_masks: 1 for real tokens, 0 for padding
    """
    packed = []
    masks = []
    current_bin = []
    current_mask = []
    
    for seq in sequences:
        # Add EOS between sequences
        seq_with_eos = seq + [eos_token_id]
        
        if len(current_bin) + len(seq_with_eos) <= max_length:
            current_bin.extend(seq_with_eos)
            current_mask.extend([1] * len(seq_with_eos))
        else:
            # Pad current bin and start new one
            pad_len = max_length - len(current_bin)
            packed.append(current_bin + [pad_token_id] * pad_len)
            masks.append(current_mask + [0] * pad_len)
            current_bin = seq_with_eos
            current_mask = [1] * len(seq_with_eos)
    
    # Handle last bin
    if current_bin:
        pad_len = max_length - len(current_bin)
        packed.append(current_bin + [pad_token_id] * pad_len)
        masks.append(current_mask + [0] * pad_len)
    
    return packed, masks

# Example: 3 short medical training examples
MAX_LENGTH = 512

# Simulate 3 tokenized examples of different lengths
np.random.seed(42)
example_1 = list(np.random.randint(100, 5000, 120))  # 120 tokens
example_2 = list(np.random.randint(100, 5000, 180))  # 180 tokens
example_3 = list(np.random.randint(100, 5000, 150))  # 150 tokens

sequences = [example_1, example_2, example_3]

# Without packing: 3 separate batches, each padded to 512
without_packing_tokens = sum(MAX_LENGTH for _ in sequences)
without_packing_real = sum(len(s) for s in sequences)
without_packing_waste = without_packing_tokens - without_packing_real

# With packing: fit into minimal bins
packed_seqs, packed_masks = pack_sequences(sequences, MAX_LENGTH)
with_packing_tokens = len(packed_seqs) * MAX_LENGTH
with_packing_real = sum(sum(m) for m in packed_masks)
with_packing_waste = with_packing_tokens - with_packing_real

print("Sequence Packing Comparison")
print("=" * 50)
print(f"\nInput sequences: {len(sequences)} examples")
print(f"  Example 1: {len(example_1)} tokens")
print(f"  Example 2: {len(example_2)} tokens")
print(f"  Example 3: {len(example_3)} tokens")
print(f"  Total real tokens: {without_packing_real}")
print(f"\nWithout packing (naive padding):")
print(f"  Batches needed:    {len(sequences)}")
print(f"  Total tokens:      {without_packing_tokens}")
print(f"  Wasted (padding):  {without_packing_waste} ({100*without_packing_waste/without_packing_tokens:.1f}%)")
print(f"\nWith sequence packing:")
print(f"  Batches needed:    {len(packed_seqs)}")
print(f"  Total tokens:      {with_packing_tokens}")
print(f"  Wasted (padding):  {with_packing_waste} ({100*with_packing_waste/with_packing_tokens:.1f}%)")
print(f"\n✅ Efficiency gain: {(1 - with_packing_tokens/without_packing_tokens)*100:.1f}% fewer forward passes")

# Visualize packing
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Without packing
ax1 = axes[0]
colors_no_pack = ['#3498db', '#e67e22', '#9b59b6']
for i, (seq, color) in enumerate(zip(sequences, colors_no_pack)):
    ax1.barh(i, len(seq), color=color, label=f'Example {i+1} ({len(seq)} tokens)')
    ax1.barh(i, MAX_LENGTH - len(seq), left=len(seq), color='#ecf0f1', edgecolor='gray', linewidth=0.5)
ax1.set_xlim(0, MAX_LENGTH)
ax1.set_title('Without Packing (3 batches, lots of padding)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Token Position')
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['Batch 1', 'Batch 2', 'Batch 3'])
ax1.legend(loc='lower right', fontsize=8)
gray_patch = mpatches.Patch(color='#ecf0f1', label='Padding (wasted)')
ax1.legend(handles=[gray_patch] + [mpatches.Patch(color=c, label=f'Ex {i+1}') for i, c in enumerate(colors_no_pack)], fontsize=8)

# With packing
ax2 = axes[1]
for i, (packed, mask) in enumerate(zip(packed_seqs, packed_masks)):
    real_tokens = sum(mask)
    pad_tokens = MAX_LENGTH - real_tokens
    ax2.barh(i, real_tokens, color='#2ecc71', label='Real tokens' if i == 0 else '')
    ax2.barh(i, pad_tokens, left=real_tokens, color='#ecf0f1', edgecolor='gray', linewidth=0.5, label='Padding' if i == 0 else '')
ax2.set_xlim(0, MAX_LENGTH)
ax2.set_title(f'With Packing ({len(packed_seqs)} batch, minimal padding)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Token Position')
ax2.set_yticks(list(range(len(packed_seqs))))
ax2.set_yticklabels([f'Batch {i+1}' for i in range(len(packed_seqs))])
ax2.legend(fontsize=8)

plt.suptitle('Sequence Packing: Eliminating Padding Waste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sequence_packing.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.3 pass@k Unbiased Estimator

In [ ]:
from math import comb

def pass_at_k(n: int, c: int, k: int) -> float:
    """
    Unbiased estimator for pass@k.
    
    From Chen et al. (2021) "Evaluating Large Language Models Trained on Code"
    
    Args:
        n: total number of samples generated
        c: number of correct samples
        k: number of samples to use (k <= n)
    
    Returns:
        Probability that at least one of k samples is correct
    
    Formula: 1 - C(n-c, k) / C(n, k)
    """
    if n - c < k:
        return 1.0  # Guaranteed to have a correct sample
    return 1.0 - comb(n - c, k) / comb(n, k)

# Demonstrate the formula
print("pass@k Unbiased Estimator")
print("=" * 50)
print("Formula: pass@k = 1 - C(n-c, k) / C(n, k)")
print()
print("Example: n=10 samples, c=3 correct")
print(f"  pass@1  = {pass_at_k(10, 3, 1):.4f} ({pass_at_k(10, 3, 1)*100:.1f}%)")
print(f"  pass@3  = {pass_at_k(10, 3, 3):.4f} ({pass_at_k(10, 3, 3)*100:.1f}%)")
print(f"  pass@5  = {pass_at_k(10, 3, 5):.4f} ({pass_at_k(10, 3, 5)*100:.1f}%)")
print(f"  pass@10 = {pass_at_k(10, 3, 10):.4f} ({pass_at_k(10, 3, 10)*100:.1f}%)")
print()
print("Why unbiased? Naive pass@k = c/n is biased (overestimates for small k)")
print(f"  Naive pass@1 = {3/10:.4f} vs unbiased = {pass_at_k(10, 3, 1):.4f}")
print(f"  Naive pass@5 = {3/10:.4f} vs unbiased = {pass_at_k(10, 3, 5):.4f}")

### 3.4 Evaluating GAS Advisor with pass@5

In [ ]:
# Simulate GAS Advisor evaluation on 10 medical test cases
# For each test case, we sample n=10 responses and check correctness

np.random.seed(42)

# Medical test cases for GAS Advisor
test_cases = [
    {"id": 1, "prompt": "Glucose 55 mg/dL, patient shaky", "correct_action": "treat_hypoglycemia"},
    {"id": 2, "prompt": "Glucose 400 with nausea/vomiting", "correct_action": "check_ketones_er"},
    {"id": 3, "prompt": "HbA1c 9.8%, patient asymptomatic", "correct_action": "medication_review"},
    {"id": 4, "prompt": "Missed insulin dose this morning", "correct_action": "take_next_dose"},
    {"id": 5, "prompt": "Non-healing foot wound 3 weeks", "correct_action": "podiatrist_referral"},
    {"id": 6, "prompt": "Glucose 65 before marathon", "correct_action": "eat_carbs_first"},
    {"id": 7, "prompt": "Sick with flu, not eating", "correct_action": "continue_insulin"},
    {"id": 8, "prompt": "Chest pain in Type 2 diabetic", "correct_action": "call_911"},
    {"id": 9, "prompt": "Glucose 280, no symptoms", "correct_action": "check_ketones_monitor"},
    {"id": 10, "prompt": "Foot numbness for 2 weeks", "correct_action": "neuropathy_evaluation"},
]

# Simulate model performance: each test case has a "true" probability of correct response
# (In practice, you'd actually run the model and evaluate outputs)
true_probs = [0.7, 0.5, 0.8, 0.9, 0.6, 0.75, 0.65, 0.85, 0.7, 0.55]

n_samples = 10  # Sample n=10 responses per test case
k_eval = 5      # Evaluate pass@5

results = []
for tc, prob in zip(test_cases, true_probs):
    # Simulate sampling n responses
    correct_samples = np.random.binomial(n_samples, prob)
    p_at_1 = pass_at_k(n_samples, correct_samples, 1)
    p_at_5 = pass_at_k(n_samples, correct_samples, k_eval)
    
    results.append({
        'id': tc['id'],
        'prompt': tc['prompt'],
        'correct_action': tc['correct_action'],
        'n_correct': correct_samples,
        'pass_at_1': p_at_1,
        'pass_at_5': p_at_5,
    })

print("GAS Advisor Evaluation: pass@k on 10 Medical Test Cases")
print("=" * 75)
print(f"{'ID':>3} {'Prompt':<35} {'Correct/10':>10} {'pass@1':>8} {'pass@5':>8}")
print("-" * 75)
for r in results:
    print(f"{r['id']:>3} {r['prompt']:<35} {r['n_correct']:>5}/10     {r['pass_at_1']:>6.3f}   {r['pass_at_5']:>6.3f}")

mean_p1 = np.mean([r['pass_at_1'] for r in results])
mean_p5 = np.mean([r['pass_at_5'] for r in results])
print("-" * 75)
print(f"{'Mean':<40} {mean_p1:>14.3f}   {mean_p5:>6.3f}")
print(f"\n✅ GAS Advisor pass@1: {mean_p1:.1%}")
print(f"✅ GAS Advisor pass@5: {mean_p5:.1%}")
print(f"   (pass@5 >> pass@1: model has correct knowledge but needs sampling)")

In [ ]:
# Visualize pass@k across different k values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: pass@k for each test case
k_values = [1, 2, 3, 5, 7, 10]
colors_tc = plt.cm.tab10(np.linspace(0, 1, len(results)))

for r, color in zip(results, colors_tc):
    pk_values = [pass_at_k(n_samples, r['n_correct'], k) for k in k_values]
    ax1.plot(k_values, pk_values, 'o-', color=color, alpha=0.7, linewidth=1.5,
             label=f"TC{r['id']}")

ax1.set_title('pass@k vs k for Each Test Case', fontsize=12, fontweight='bold')
ax1.set_xlabel('k (number of samples)')
ax1.set_ylabel('pass@k')
ax1.legend(fontsize=7, ncol=2)
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1.05)

# Plot 2: Mean pass@k
mean_pk = [np.mean([pass_at_k(n_samples, r['n_correct'], k) for r in results]) for k in k_values]
ax2.plot(k_values, mean_pk, 'bo-', linewidth=2.5, markersize=8, label='Mean pass@k')
ax2.fill_between(k_values, 
                  [np.min([pass_at_k(n_samples, r['n_correct'], k) for r in results]) for k in k_values],
                  [np.max([pass_at_k(n_samples, r['n_correct'], k) for r in results]) for k in k_values],
                  alpha=0.2, color='blue', label='Min-Max range')
ax2.axhline(y=0.8, color='green', linestyle='--', alpha=0.7, label='Target: 80%')
ax2.set_title('Mean pass@k: GAS Advisor Performance', fontsize=12, fontweight='bold')
ax2.set_xlabel('k (number of samples)')
ax2.set_ylabel('Mean pass@k')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.suptitle('GAS Advisor: pass@k Evaluation on Medical Test Cases', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pass_at_k.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.5 SFT Training Configuration for GAS

In [ ]:
# SFT training configuration for GAS Advisor base model
sft_config = '''
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset

# GAS SFT configuration
training_args = SFTConfig(
    output_dir="./gas_advisor_sft",
    
    # Training
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # Effective batch = 32
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    
    # Efficiency
    bf16=True,
    packing=True,                    # Sequence packing!
    max_length=2048,
    
    # Masking
    dataset_text_field="text",
    # completion_only_masking handled by DataCollatorForCompletionOnlyLM
    
    # Logging
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
)

# Use DataCollatorForCompletionOnlyLM for masking
from trl import DataCollatorForCompletionOnlyLM
response_template = "[/INST]"  # Llama-style instruction template
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()
'''

if not TEST_MODE:
    print("SFT Training Configuration for GAS Advisor:")
    print(sft_config)
else:
    print("TEST_MODE: SFT config shown (not executing)")

print("\nKey SFT best practices applied:")
print("  ✅ packing=True          → sequence packing, no padding waste")
print("  ✅ completion_only_mask  → only train on response tokens")
print("  ✅ cosine LR schedule    → smooth learning rate decay")
print("  ✅ bf16=True             → memory efficient training")
print("  ✅ gradient_accumulation → larger effective batch size")

---
## 🏥 4. GAS Connection

### SFT in the Guardian Angel System

SFT is the **first training stage** for the GAS Advisor Agent. Before DPO or GRPO alignment, the base model must learn the medical domain through supervised fine-tuning.

```
GAS Training Pipeline:

Stage 1: SFT (this chapter)
  Base LLM → Fine-tune on medical Q&A
  Techniques: completion masking, sequence packing
  
Stage 2: DPO/SimPO (Chapter 6/8)
  SFT model → Align with safe medical advice
  
Stage 3: GRPO (Chapter 7)
  Aligned model → Improve reasoning quality
  
Stage 4: Deployment (Chapter 11)
  Final model → Serve 5 agents concurrently
```

**These techniques ensure the GAS Advisor model is trained efficiently.**

**Completion masking** is critical for GAS because:
- Medical prompts are long (patient history, vitals, medications)
- Without masking, the model wastes capacity learning to reproduce prompts
- With masking, 100% of gradient updates improve response quality

**pass@5 evaluation** is used for GAS because:
- Medical advice often has multiple valid approaches
- We want to know if the model *can* generate correct advice, not just if it does on the first try
- In production, the Safety Critic filters responses, so pass@5 better reflects real-world performance

---
## ✅ 5. Checkpoint

Test your understanding:

**Q1: Why do we set prompt token labels to -100 in completion-only masking?**  
A: PyTorch's CrossEntropyLoss ignores positions with label=-100. By masking prompt tokens, we ensure the loss is only computed on response tokens, preventing the model from wasting gradient updates on learning to reproduce the prompt.

**Q2: How does sequence packing improve training efficiency?**  
A: Instead of padding each sequence to max_length (wasting computation on padding tokens), packing concatenates multiple short sequences into one long sequence. This can reduce the number of forward passes by 50-70% for datasets with variable-length examples.

**Q3: Why is the naive pass@k estimator (c/n) biased?**  
A: The naive estimator doesn't account for the combinatorial probability of selecting k samples from n. The unbiased estimator 1 - C(n-c,k)/C(n,k) correctly computes the probability that at least one of k randomly selected samples is correct.

**Q4: When would pass@5 be more informative than pass@1 for GAS?**  
A: pass@5 is more informative when the model has the correct knowledge but is inconsistent. If pass@1=40% but pass@5=90%, the model knows the right answer but doesn't always generate it first — the Safety Critic can help by selecting the best of multiple samples.

**Q5: What is the correct order of training stages for the GAS Advisor?**  
A: (1) SFT on medical Q&A to learn domain knowledge, (2) DPO/SimPO to align with safe advice preferences, (3) GRPO to improve step-by-step reasoning quality. Each stage builds on the previous one.